In [1]:
# Cell 1 — Install deps
!pip install -q -U transformers accelerate peft huggingface_hub bitsandbytes
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 86.2 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 47.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 47.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 95.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 36.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0

In [2]:
# Cell 2 — Auth
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

In [3]:
# Cell 3 — Config — EDIT THESE
BASE_MODEL_ID   = "microsoft/Phi-3.5-mini-instruct"   # base you fine-tuned from
ADAPTER_REPO_ID = "TeslaInch/scd-phi35-adapter-v8"    # your LoRA adapter on HF
MERGED_REPO_ID  = "TeslaInch/phi-3.5-mini-SCD"  # where merged model goes

In [4]:
# Cell 4 — Load base model in fp16 (not 4-bit — merging needs full precision math)
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Loading base model...")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    #trust_remote_code=False,  # Phi-3.5 needs this
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, trust_remote_code=True)

Loading base model...


config.json:   0%|          | 0.00/3.45k [00:00<?, ?B/s]

[transformers] This model config has set a `rope_parameters['original_max_position_embeddings']` field, to be used together with `max_position_embeddings` to determine a scaling factor. Please set the `factor` field of `rope_parameters`with this ratio instead -- we recommend the use of this field over `original_max_position_embeddings`, as it is compatible with most model architectures.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/16.3k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

...

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

In [5]:
# Cell 5 — Load and merge the adapter
from peft import PeftModel

print("Loading adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO_ID)

print("Merging...")
model = model.merge_and_unload()   # bakes LoRA weights into base, returns plain model
print("Merge complete.")

Loading adapter...


adapter_config.json:   0%|          | 0.00/1.41k [00:00<?, ?B/s]

adapter_model.safetensors: reconstructing file:   0%|          |  0.00B / 12.6MB            

adapter_model.safetensors: downloading bytes:           |  0.00B            

Merging...
Merge complete.


In [9]:
# Cell 6 — Sanity check BEFORE pushing anything

# 1. Define a realistic clinical question and mock context from your vector DB
system_instruction = (
    "You are a medical AI assistant specialised in sickle cell disease. "
    "Answer clinical questions accurately and concisely. "
    "If you are uncertain, say so clearly rather than guessing."
)

mock_context = (
    "Clinical Guidelines for Sickle Cell Disease:\n"
    "Dactylitis (hand-foot syndrome) is often the first manifestation of pain in children with sickle cell anemia (HbSS). "
    "It typically presents in infants and toddlers (aged 6 months to 4 years) with fever, swelling, and tenderness of the hands and feet. "
    "Immediate evaluation for sepsis and empiric IV antibiotics is required if fever is present due to functional asplenia."
)

clinical_case = (
    "Clinical case:\n"
    "A 4-year-old boy with HbSS presents to the emergency department with a fever of 39°C and bilateral swollen, tender hands and feet.\n\n"
    "Question: What is the most likely diagnosis and what is the immediate priority for management?"
)

# 2. Build the exact chat format Phi-3.5-mini expects
messages = [
    {"role": "system", "content": system_instruction},
    {"role": "user", "content": f"Context:\n{mock_context}\n\n{clinical_case}"}
]

# Apply the chat template (this automatically adds <|system|>, <|user|>, <|end|> etc.)
prompt = tokenizer.apply_chat_template(
    messages, 
    tokenize=False, 
    add_generation_prompt=True
)

# 3. Tokenize and Generate
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

print("Generating response...")
with torch.no_grad():
    out = model.generate(
        **inputs, 
        max_new_tokens=200, 
        do_sample=False,        # Use greedy decoding for a deterministic sanity check
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id
    )

# 4. Decode and extract only the assistant's response (ignoring the prompt)
# out[0][inputs.input_ids.shape[1]:] slices off the input tokens so you only see the new generation
generated_tokens = out[0][inputs.input_ids.shape[1]:]
response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

print("\n--- MODEL RESPONSE ---")
print(response)
print("----------------------")


[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generating response...

--- MODEL RESPONSE ---
The most likely diagnosis for this 4-year-old boy with HbSS presenting with fever, bilateral swollen, and tender hands and feet is dactylitis, also known as hand-foot syndrome. This condition is a common manifestation of pain in children with sickle cell anemia, particularly in infants and toddlers. The clinical presentation of dactylitis typically includes fever, swelling, and tenderness of the hands and feet, which are often accompanied by a decrease in hemoglobin levels and an increase in reticulocyte count.

The immediate priority for management in this case is to evaluate for sepsis and initiate empiric IV antibiotics. This is crucial due to the functional asplenia associated with sickle cell anemia, which increases the risk of bacterial infections, including sepsis. The presence of fever in a child
----------------------


In [11]:
SAVE_DIR = "/kaggle/working/merged-model"
model.save_pretrained(SAVE_DIR, safe_serialization=True)
tokenizer.save_pretrained(SAVE_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/merged-model/tokenizer_config.json',
 '/kaggle/working/merged-model/chat_template.jinja',
 '/kaggle/working/merged-model/tokenizer.json')

In [12]:
import json
with open(f"{SAVE_DIR}/tokenizer_config.json") as f:
    cfg = json.load(f)
print("chat_template" in cfg)  # should print True

False


In [13]:

print("Saved locally. Files:")
import os
print(os.listdir(SAVE_DIR))

Saved locally. Files:
['tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'generation_config.json', 'model.safetensors', 'config.json']


In [15]:
# Cell 8 — Push to HF Hub
from huggingface_hub import HfApi, create_repo

create_repo(MERGED_REPO_ID, exist_ok=True, private=False)

model.push_to_hub(MERGED_REPO_ID)
tokenizer.push_to_hub(MERGED_REPO_ID)

print(f"Pushed to https://huggingface.co/{MERGED_REPO_ID}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Pushed to https://huggingface.co/TeslaInch/phi-3.5-mini-SCD


In [ ]:
from huggingface_hub import HfApi, create_repo

VECTOR_DB_REPO_ID = "TeslaInch/SCD-vectorDB"

create_repo(VECTOR_DB_REPO_ID, repo_type="dataset", exist_ok=True, private=False)

api = HfApi()
api.upload_folder(
    folder_path="chroma_db",          # your local Chroma persistence dir
    repo_id=VECTOR_DB_REPO_ID,
    repo_type="dataset",
)

print(f"Pushed to https://huggingface.co/datasets/{VECTOR_DB_REPO_ID}")